# 01 · A first spiking network

Build spiking networks by hand with py-spinnaker2: one neuron, then neurons driven by input spikes, then a small network. Everything runs on the Brian2 simulator; participants with cluster access run the last network on the SpiNNaker2 chip in section 5.

Work in pairs. Exercises are marked **Try**.

## 0. Setup

`USE_HARDWARE = False` runs on Brian2. Leave it like this until section 5.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from spinnaker2 import brian2_sim, helpers, snn

USE_HARDWARE = False


def backend():
    """A fresh backend for every run: Brian2, or one 48-chip SpiNNaker2 board."""
    if USE_HARDWARE:
        from spinnaker2 import hardware

        return hardware.SpiNNcloud48NodeBoard()
    return brian2_sim.Brian2Backend()


def raster(ax, spikes, n, title, color="royalblue"):
    """Spike raster: one row per neuron, one tick per spike."""
    idx, t = helpers.spike_times_dict_to_arrays(spikes)
    ax.plot(t, idx, "|", ms=6, mew=1.2, color=color)
    ax.set(xlabel="time step (1 step = 1 ms)", ylabel="neuron index", ylim=(-0.5, n - 0.5), title=title)

## 1. One neuron with a constant input

The `lif` neuron of SpiNNaker2 updates its membrane potential once per time step:

$$v[t] = \alpha \, v[t-1] + I_\text{syn}[t] + i_\text{offset}$$

- `alpha_decay` ($\alpha$): how much of the potential is kept per step (0.9 = 10 % leaks away)
- `i_offset`: a constant input added every step
- `threshold`: when $v$ exceeds it, the neuron spikes and $v$ is reset (here by subtracting the threshold)

The cell below runs one neuron for 50 steps and plots its membrane potential. The recorded potential is the value after a possible reset, so the moment of crossing the threshold shows up as a drop, marked with a red triangle.

In [ ]:
params = {"threshold": 1.0, "alpha_decay": 0.9, "i_offset": 0.15, "reset": "reset_by_subtraction"}

neuron = snn.Population(size=1, neuron_model="lif", params=params, name="neuron", record=["spikes", "v"])
net = snn.Network("one_neuron")
net.add(neuron)
backend().run(net, 50)

v = neuron.get_voltages()[0]
spikes = neuron.get_spikes()[0]
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(v, label="membrane potential v")
ax.axhline(params["threshold"], ls="--", c="0.5", label="threshold")
ax.plot(spikes, [params["threshold"]] * len(spikes), "rv", label="spike")
ax.set(xlabel="time step (1 step = 1 ms)", ylabel="v (units of threshold)")
ax.legend(loc="lower right")
plt.show()
print(f"{len(spikes)} spikes in 50 steps")

**Try:**

1. Set `i_offset` to 0.05. Why does the neuron never spike? *Hint:* without spikes, $v$ approaches $i_\text{offset} / (1 - \alpha)$.
2. What is the smallest `i_offset` that makes the neuron spike with `alpha_decay = 0.9`? Check your answer.
3. Increase `i_offset` step by step. How does the number of spikes change? This is the neuron's input–output curve, the spiking counterpart of an activation function.

## 2. Input spikes and synapses

Neurons usually get their input as spikes from other neurons. A `spike_list` population sends spikes at given time steps. A `Projection` connects it to the `lif` neuron; each connection is `[pre neuron, post neuron, weight, delay]`.

On the chip, `lif` weights are integers from −15 to 15 and delays from 0 to 7 steps, so the threshold is set to 10 here instead of 1. A spike arrives `delay + 1` steps after it was sent.

In [ ]:
input_times = {0: [5, 7, 9], 1: [30], 2: [32]}  # input neuron -> spike times

inp = snn.Population(size=3, neuron_model="spike_list", params=input_times, name="input")
post = snn.Population(
    size=1,
    neuron_model="lif",
    params={"threshold": 10.0, "alpha_decay": 0.8, "reset": "reset_by_subtraction"},
    name="post",
    record=["spikes", "v"],
)
weight = 5  # same weight for all three input neurons, half the threshold of 10
conns = [[i, 0, weight, 0] for i in range(3)]  # pre, post, weight, delay
net = snn.Network("synapses")
net.add(inp, post, snn.Projection(pre=inp, post=post, connections=conns))
backend().run(net, 50)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(post.get_voltages()[0], label="membrane potential v")
ax.axhline(10.0, ls="--", c="0.5", label="threshold")
post_spikes = post.get_spikes()[0]
ax.plot(post_spikes, [10.0] * len(post_spikes), "rv", label="spike of the post neuron")
for i, ts in input_times.items():
    ax.plot(ts, [-1] * len(ts), "|", ms=10, mew=2, label=f"input neuron {i}")
ax.set(xlabel="time step (1 step = 1 ms)", ylabel="v (threshold = 10)")
ax.legend(loc="upper right", fontsize=8)
plt.show()
print("post neuron spikes at:", post_spikes)

**Try:**

1. One input spike is not enough to reach the threshold. Which input makes the neuron spike: the three spikes of neuron 0, or the two spikes of neurons 1 and 2? Why?
2. Change `alpha_decay` to 0.95 and to 0.5. What happens, and why does the leak matter for integrating spikes that arrive at different times?
3. Make one connection inhibitory (negative weight). Where does the potential go?

## 3. A small network

20 input neurons fire at random, like a sensor; 10 `lif` neurons receive input from a random subset of them. The input spike trains are generated with NumPy and fed in as spike times, so the input is identical on Brian2 and on the chip.

In [ ]:
rng = np.random.default_rng(seed=1)
n_in, n_out, steps = 20, 10, 200

rate = 0.05  # probability of a spike per input neuron and step (50 Hz at 1 ms per step)
input_trains = {i: list(np.nonzero(rng.random(steps) < rate)[0]) for i in range(n_in)}

p_connect = 0.3  # probability that an input neuron connects to an output neuron
weight = 4  # integer, chip range -15..15
threshold = 10.0
conns = [[i, j, weight, 0] for i in range(n_in) for j in range(n_out) if rng.random() < p_connect]


def build_network():
    inp = snn.Population(size=n_in, neuron_model="spike_list", params=input_trains, name="input")
    out = snn.Population(
        size=n_out,
        neuron_model="lif",
        params={"threshold": threshold, "alpha_decay": 0.9, "reset": "reset_by_subtraction"},
        name="output",
        record=["spikes"],
    )
    net = snn.Network("small_network")
    net.add(inp, out, snn.Projection(pre=inp, post=out, connections=conns))
    return net, out


net, out = build_network()
backend().run(net, steps)
out_spikes = out.get_spikes()

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.5))
raster(a1, input_trains, n_in, "input spikes")
raster(a2, out_spikes, n_out, "output spikes", color="crimson")
plt.tight_layout()
plt.show()

n_in_spikes = sum(len(s) for s in input_trains.values())
n_out_spikes = sum(len(s) for s in out_spikes.values())
fan_out = len(conns) / n_in
print(f"{len(conns)} connections, {n_in_spikes} input spikes, {n_out_spikes} output spikes")
print(f"synaptic events (input spikes x average fan-out): about {n_in_spikes * fan_out:.0f}")

**Try:**

1. Halve the input `rate`. How does the output activity change, and how many synaptic events are left?
2. Change `threshold` or `weight` so that only a few output neurons spike. This is a sparse network: little activity, little work.
3. Look at the numbers printed above: every input spike triggers one synaptic event per connection. On SpiNNaker2, the work and the energy grow with the number of events, not with the number of neurons.

## 4. What you built

- `snn.Population`: a group of neurons of one model (`lif`) or an input (`spike_list`)
- `snn.Projection`: the connections between two populations, as a list of `[pre, post, weight, delay]`
- `snn.Network`: everything that runs together; `run(net, steps)` simulates it

In hands-on 2, the NIR import (`s2_nir.from_nir`) builds exactly these objects automatically from a PyTorch model.

## 5. Run the network on the SpiNNaker2 chip (cluster participants)

Set `USE_HARDWARE = True` below and run the cell. It runs the network from section 3 on one 48-chip board and compares the output with Brian2. A chip run takes about one minute, mostly for powering up and booting the board; if all boards are busy, the run waits automatically.

Participants without cluster access: read along, the expected result is shown in the workshop.

In [ ]:
USE_HARDWARE = False  # set to True on the cluster

brian2_spikes = out_spikes
net, out = build_network()
backend().run(net, steps)
chip_spikes = out.get_spikes()

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.5))
raster(a1, brian2_spikes, n_out, "Brian2", color="crimson")
raster(a2, chip_spikes, n_out, "SpiNNaker2" if USE_HARDWARE else "Brian2 again (USE_HARDWARE is False)", color="darkgreen")
plt.tight_layout()
plt.show()

same = all(list(brian2_spikes.get(i, [])) == list(chip_spikes.get(i, [])) for i in range(n_out))
print("identical spike times on both backends" if same else "spike times differ")